In [0]:
# Cargar la tabla en un DataFrame de **Spark**
df = spark.table("data.client_transaction_orders")

In [0]:
# Muestra las primeras 10 filas en formato de tabla interactiva
display(df.limit(10))

In [0]:
# Contar el número total de filas en el DataFrame
total_filas = df.count()

print(f"El número total de filas es: {total_filas}")

# Contar el número de clientes únicos
total_clientes_unicos = df.select("cliente_id").distinct().count()

print(f"El número total de clientes únicos es: {total_clientes_unicos}")

In [0]:

from pyspark.sql.functions import countDistinct

# Agrupar por canal y contar los clientes distintos
clientes_por_canal = df.groupBy("canal_pedido_cd") \
    .agg(
        countDistinct("cliente_id").alias("numero_de_clientes")
    )

# Muestra el resultado
display(clientes_por_canal)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import sum, avg, percentile_approx

# Agrupar por canal de pedido y calcular las métricas solicitadas
metricas_por_canal = df.groupBy("canal_pedido_cd") \
    .agg(
        sum("facturacion_usd_val").alias("ventas_totales"),
        avg("facturacion_usd_val").alias("ventas_promedio"),
        percentile_approx("facturacion_usd_val", 0.5).alias("ventas_mediana")
    )

# Mostrar los resultados en una tabla interactiva
display(metricas_por_canal)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import sum, countDistinct, desc

# Filtra los datos para excluir el canal 'DIGITAL'
df_no_digital = df.filter(df.canal_pedido_cd != 'DIGITAL')

# Agrupa por país y calcula la facturación total y el número de clientes únicos
metricas_por_pais = df_no_digital.groupBy("pais_cd") \
    .agg(
        sum("facturacion_usd_val").alias("facturacion_total_usd"),
        countDistinct("cliente_id").alias("numero_de_clientes")
    ) \
    .orderBy(desc("facturacion_total_usd"))

# Visualiza el resultado
display(metricas_por_pais)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import sum, countDistinct, desc

# Agrupa por región comercial y calcula la facturación y el número de clientes
metricas_por_region = df_no_digital.groupBy("region_comercial_txt") \
    .agg(
        sum("facturacion_usd_val").alias("facturacion_total_usd"),
        countDistinct("cliente_id").alias("numero_de_clientes")
    ) \
    .orderBy(desc("facturacion_total_usd"))

# Visualiza el resultado
display(metricas_por_region)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.functions import sum

# 1. Calcular la facturación total por cliente (si no lo has hecho)
facturacion_por_cliente = df.groupBy("cliente_id") \
    .agg(sum("facturacion_usd_val").alias("facturacion_total_cliente"))

# 2. Calcular los umbrales para el top 30%, 20% y 10%
umbrales = facturacion_por_cliente.approxQuantile(
    "facturacion_total_cliente", [0.70, 0.80, 0.90], 0.01
)
umbral_top_30 = umbrales[0] # Percentil 70
umbral_top_20 = umbrales[1] # Percentil 80
umbral_top_10 = umbrales[2] # Percentil 90

# 3. Convertir a pandas para el gráfico
pandas_clientes_df = facturacion_por_cliente.toPandas()

# 4. Crear el histograma
sns.set_style("whitegrid")
plt.figure(figsize=(12, 7))
sns.histplot(data=pandas_clientes_df, x='facturacion_total_cliente', bins=50, kde=True)

# 5. Añadir las líneas de percentiles superiores al gráfico
plt.axvline(x=umbral_top_30, color='gold', linestyle='--', label=f'Top 30% (desde ${umbral_top_30:,.2f})')
plt.axvline(x=umbral_top_20, color='darkorange', linestyle='--', label=f'Top 20% (desde ${umbral_top_20:,.2f})')
plt.axvline(x=umbral_top_10, color='red', linestyle='--', label=f'Top 10% (desde ${umbral_top_10:,.2f})')

# Añadir títulos, etiquetas y la leyenda
plt.title('Distribución de Facturación por Cliente no digitales con Umbrales Superiores', fontsize=16)
plt.xlabel('Facturación Total por Cliente (USD)', fontsize=12)
plt.ylabel('Número de Clientes', fontsize=12)
plt.legend() # Muestra las etiquetas de las líneas

# Muestra el gráfico
plt.show()

In [0]:
from pyspark.sql.functions import sum, desc

# 1. Filtrar los datos para excluir el canal 'DIGITAL'
df_no_digital = df.filter(df.canal_pedido_cd != 'DIGITAL')

# 2. Calcular cuántos clientes representan el 20%
total_clientes_no_digital = df_no_digital.select("cliente_id").distinct().count()
limite_20_porciento = int(total_clientes_no_digital * 0.20)

# 3. Agrupar por cliente, sumar la facturación, ordenar y tomar el 20% superior
top_20_porciento_clientes = df_no_digital.groupBy("cliente_id") \
    .agg(sum("facturacion_usd_val").alias("facturacion_total")) \
    .orderBy(desc("facturacion_total")) \
    .limit(limite_20_porciento)

# Visualizar el resultado
print(f"Mostrando el top 20% de clientes no digitales (aproximadamente {limite_20_porciento} clientes):")
display(top_20_porciento_clientes)

In [0]:
from pyspark.sql.functions import trunc

# Crear una columna 'mes_pedido' truncando la fecha al primer día del mes
df_con_mes = df.withColumn("mes_pedido", trunc("fecha_pedido_dt", "month"))

# Agrupar por mes y sumar la facturación
facturacion_mensual = df_con_mes.groupBy("mes_pedido") \
                                .agg(sum("facturacion_usd_val").alias("facturacion_total_mes")) \
                                .orderBy("mes_pedido")

# El comando display() te permitirá graficar esto fácilmente
display(facturacion_mensual)